# Detect Columns by using ML

---

In [1]:
import numpy as np
import pandas as pd
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functions import *
import scipy
import signal
import io
import cv2
import os
import glob
from scipy.signal import find_peaks
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from scipy.ndimage import uniform_filter1d

**Machine learning (ML)** is a branch of artificial intelligence. It allows a computer to learn from data and to improve decision making with experience.

---

## Using Random Forest

**L'Arbre de Décision (Decision Tree) :**
Imagine un jeu de "Qui est-ce ?". L'algorithme pose une série de questions par oui/non sur les caractéristiques (features) de tes données pour arriver à une conclusion. Par exemple : "La variance de cette colonne est-elle supérieure à 450 ?" -> Si oui, on va à droite ; si non, on va à gauche.

Le Random Forest repose sur l'apprentissage d'ensemble (Ensemble Learning), et plus précisément sur une technique appelée Bagging (Bootstrap Aggregating). L'algorithme va créer une "forêt" composée de dizaines, voire de centaines d'arbres de décision.

Pour classer une nouvelle colonne (Normale vs Défectueuse), la forêt fait passer les données de la colonne dans tous ses arbres. Chaque arbre vote. La classe qui obtient la majorité des votes l'emporte.

In [2]:
def load_img_strict(path):
    """
    Lecture universelle et stricte pour garantir que l'entraînement 
    et l'inférence voient EXACTEMENT la même chose.
    """
    # IMREAD_ANYDEPTH force la lecture en 16-bits brut (0 à 65535) sans compression 8-bits
    img = cv2.imread(path, cv2.IMREAD_ANYDEPTH)
    
    # Sécurité au cas où l'image aurait été sauvegardée avec 3 canaux (couleur)
    if img.ndim == 3:
        img = img[:, :, 0]
        
    return img

### Features that we use : 

- Moyenne `mean` : Valeur moyenne des intensités de la colonne. Luminosité globale de la colonne. Si trop ou trop basse peut indiquer anomalie.
- Ecart-type `std` : Dispersion autour de la moyenne. Colonne peut-être *noisy* si l'écart-type est très élevé.
- Min et Max : Si la colonne à des valeurs super hautes ou super basses c'est que l'amplificateur est défectueux (dixit Phlypo)
- Range `range` : Différence entre max-min
- Médian `median` : Valeur centrale. Comparaison avec la moyenne intéressant.
- Quantile `q25` `q75` : Quartile.
- Skewness `skewness` : Asymétrie 
- Kurtosis `kurtosis`: Applatissement 
- Energie `energy` : Energie d'un signal (dixit Signals and Systems)
- Entropie `entropy` : J'ai pas vrmt compris / Mesure du désordre ou de l'incertitude dans la distribution des intensités.
- Nombres de pics `num_peaks` : les pics dans la colonne si gros peut-être *fragmented*
- Moyenne du gradiant `mean_gradient` : Moyenne des différences entre pixels consécutifs

#### A faire : 
- Différence Spatiale (un peu comme mes autres méthodes)
- Différence Temporelle (pour les blinking)

In [3]:
import numpy as np
from scipy.ndimage import uniform_filter1d

def extract_image_features_vectorized(img_prev, img_curr, img_next):
    # 1. Conversion en float64 pour toute l'image d'un coup
    img_prev = img_prev.astype(np.float64)
    img_curr = img_curr.astype(np.float64)
    img_next = img_next.astype(np.float64)
    
    height, width = img_curr.shape[:2]
    
    # DÉTECTION DE LA RÉSOLUTION (Pour la fenêtre du LT_moy)
    if height < 700: 
        taille_fenetre = 18
    elif height < 1050: 
        taille_fenetre = 31
    else: 
        taille_fenetre = 24

    # --- Statistiques Globales ---
    col_mean = np.mean(img_curr, axis=0) # Vecteur avec la moyenne de chaque colonne
    col_energy = np.sum(img_curr ** 2, axis=0) / height
    
    # =========================================================================
    # 1. LT_moy (Fenêtre glissante ultra-rapide)
    # =========================================================================
    # uniform_filter1d fait la tendance locale pour TOUTES les colonnes instantanément
    tendance_locale = uniform_filter1d(col_mean, size=taille_fenetre, mode='reflect')
    
    # Calcul de l'écart-type local via la variance : V(X) = E(X^2) - E(X)^2
    mean_sq = np.mean(img_curr ** 2, axis=0)
    tendance_sq = uniform_filter1d(mean_sq, size=taille_fenetre, mode='reflect')
    std_locale = np.sqrt(np.maximum(tendance_sq - tendance_locale**2, 0))
    
    ecart_lt_moy = np.abs(col_mean - tendance_locale)
    ratio_lt_moy = ecart_lt_moy / (std_locale + 1e-5)
    
    # =========================================================================
    # 2. Spatial & Temporel
    # =========================================================================
    # Décalage du vecteur pour comparer avec la colonne de gauche et de droite
    left_neighbor = np.roll(col_mean, 1)
    right_neighbor = np.roll(col_mean, -1)
    neighbor_mean = (left_neighbor + right_neighbor) / 2.0
    
    # Correction des extrêmes (bords de l'image)
    neighbor_mean[0] = col_mean[1]
    neighbor_mean[-1] = col_mean[-2]
    
    spatial_diff = np.abs(col_mean - neighbor_mean)
    
    # Contexte temporel
    mean_prev = np.mean(img_prev, axis=0)
    mean_next = np.mean(img_next, axis=0)
    diff_temp_absolue = np.abs(col_mean - mean_prev)
    scintillement_temporel = np.abs(col_mean - ((mean_prev + mean_next) / 2.0))

    # --- Assemblage Final (Seulement nos 7 super-features !) ---
    features_matrix = np.column_stack((
        ecart_lt_moy, ratio_lt_moy,
        spatial_diff, diff_temp_absolue, scintillement_temporel,
        # col_mean, col_energy
    ))
    
    return features_matrix

def process_single_image(img_num, images_list, json_data):
    """Prépare les labels et lance l'extraction de l'image entière"""
    img_curr = images_list[img_num]
    img_prev = images_list[img_num - 1] if img_num > 0 else img_curr
    img_next = images_list[img_num + 1] if img_num < (len(images_list) - 1) else img_curr
    
    width = img_curr.shape[1]
    
    # Extraction vectorisée
    features_matrix = extract_image_features_vectorized(img_prev, img_curr, img_next)
    
    # Création rapide des labels (y)
    defects = set(get_defect_coordinates(json_data, img_num))
    y_img = [1 if x in defects else 0 for x in range(width)]
    
    return features_matrix.tolist(), y_img

In [4]:
def build_dataset(images, json_data):
    print(f"Lancement de l'extraction sur {len(images)} images en parallèle...")
    
    # n_jobs=-1 (utilisation de tous les coeurs)
    # return_as="generator" permet à tqdm de se mettre à jour en temps réel
    result_generator = Parallel(n_jobs=-1, require="sharedmem", return_as="generator")(
        delayed(process_single_image)(img_num, images, json_data) 
        for img_num in range(len(images))
    )
    
    X = []
    y = []
    
    # On enveloppe le générateur avec tqdm pour la barre de progression
    for X_img, y_img in tqdm(result_generator, total=len(images), desc="Extraction Multicoeur"):
        X.extend(X_img)
        y.extend(y_img)
        
    return np.array(X), np.array(y)

### Lezz gooo

In [5]:
def load_images(folder='train', type='VGA', sequence='sequence_1', dyn='low dyn with columns 1'):
    """
    Version ultra-stricte insérée directement dans le notebook.
    Écrase la fonction de functions.py pour forcer le 16-bits mono.
    """
    images = []
    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')
    fichiers_trouves = sorted(glob.glob(chemin_recherche))
    
    for image_path in fichiers_trouves:
        # IMREAD_ANYDEPTH force la lecture en 16-bits d'origine et évite le piège des 3 canaux couleur
        img = cv2.imread(image_path, cv2.IMREAD_ANYDEPTH)
        
        if img is None:
            print(f"⚠️ Impossible de charger : {image_path}")
            continue
            
        # Sécurité ultime : si l'image a quand même 3 dimensions, on ne garde qu'un canal
        if img.ndim == 3:
            img = img[:, :, 0]
            
        images.append(img)
        
    return images

In [6]:
import numpy as np
import pandas as pd
import xgboost as xgb
import traceback
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm # Import modifié pour éviter l'erreur IProgress !

# =========================================================================
# 1. PARAMÈTRES DU SUPER DATASET
# =========================================================================
CAMERA_TYPE = 'HD'  # Change ceci en 'HD', 'VGA' ou 'SXGA'

# Création automatique du nouveau dossier demandé
os.makedirs('new_models', exist_ok=True)

dyn_mapping = {
    1: 'low dyn with columns 1',
    2: 'low dyn with columns 2',
    3: 'low dyn with columns 3'
}

# On prépare deux listes distinctes pour le Train et le Test
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

noms_colonnes = [
    'ecart_lt_moy', 'ratio_lt_moy', 
    'spatial_diff', 'diff_temp_absolue', 'scintillement_temporel', 
    # 'mean', 'energy'
]

# =========================================================================
# 2. BOUCLE D'EXTRACTION AUTOMATIQUE (Séparation Séquences 1&2 vs 3)
# =========================================================================
for seq in [1, 2, 3]:
    for config in [1, 2, 3]:
        print(f"\n🚀 Lancement : {CAMERA_TYPE} | Séquence {seq} | Configuration {config}")
        
        sequence_name = f'sequence_{seq}'
        dyn_name = dyn_mapping[config]
        
        dossier_json = r"C:\Users\alexc\Documents\Data_challenge\Data_Challenge\results"
        json_path = os.path.join(dossier_json, f"{CAMERA_TYPE}_{sequence_name}_config_{config}.json") 
        
        try:
            images = load_images(type=CAMERA_TYPE, sequence=sequence_name, dyn=dyn_name)
            json_data = load_json(json_path)
            
            X_batch, y_batch = build_dataset(images, json_data)
            
            # --- LE SCRIPT SE PARE ICI ---
            if seq in [1, 2]:
                X_train_list.append(X_batch)
                y_train_list.append(y_batch)
            else:  # seq == 3
                X_test_list.append(X_batch)
                y_test_list.append(y_batch)
            
        except Exception as e:
            print(f"⚠️ Erreur pour Seq {seq} / Config {config} : {e}")
            continue

# Fusion des blocs
X_train_raw = np.vstack(X_train_list)
y_train_raw = np.concatenate(y_train_list)

X_test_raw = np.vstack(X_test_list)
y_test_raw = np.concatenate(y_test_list)

print(f"\n✅ Extraction terminée !")
print(f"Forme brute Train (Seq 1 & 2) : {X_train_raw.shape}")
print(f"Forme brute Test (Seq 3)       : {X_test_raw.shape}")
# ==# =========================================================================
# 3. RÉÉQUILIBRAGE DU DATASET D'ENTRAÎNEMENT UNIQUEMENT
# =========================================================================
indices_defauts = np.where(y_train_raw == 1)[0]
indices_sains = np.where(y_train_raw == 0)[0]

# On garde 2 fois plus de colonnes saines que de défauts
nb_sains_a_garder = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_sains_a_garder, replace=False)

indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_train_balanced = X_train_raw[indices_finaux]
y_train_balanced = y_train_raw[indices_finaux]

print(f"Train équilibré : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} saines.")

# =========================================================================
# 4. ENTRAÎNEMENT ET SAUVEGARDE DANS 'new_models'
# =========================================================================
# Conversion en DataFrame Pandas
X_train = pd.DataFrame(X_train_balanced, columns=noms_colonnes)
X_test = pd.DataFrame(X_test_raw, columns=noms_colonnes)
y_train = y_train_balanced
y_test = y_test_raw

print(f"Shape finale X_train : {X_train.shape}")
print(f"Shape finale X_test  : {X_test.shape}")

# Calcul du ratio de poids pour XGBoost
ratio_poids = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

clf = xgb.XGBClassifier(
    n_estimators=150,
    scale_pos_weight=ratio_poids, 
    random_state=42,
    tree_method="hist",
    device="cpu"  # Mets "cuda" si ton GPU est disponible
)

# Entraînement sur Séquences 1 & 2
clf.fit(X_train, y_train)

# Prédiction sur la Séquence 3 (Inconnue du modèle)
y_pred = clf.predict(X_test)

print("\n📊 RÉSULTATS DU MODÈLE XGBOOST (Évalué uniquement sur la Séquence 3) :")
print(classification_report(y_test, y_pred))

# Sauvegarde dans le NOUVEAU dossier demandé
model_path = f'new_models/xgboost_{CAMERA_TYPE.lower()}_baseline.json'
clf.save_model(model_path)
print(f"💾 Succès : Modèle sauvegardé sous {model_path} !")


🚀 Lancement : HD | Séquence 1 | Configuration 1
Lancement de l'extraction sur 300 images en parallèle...


Extraction Multicoeur: 100%|██████████| 300/300 [00:02<00:00, 112.72it/s]



🚀 Lancement : HD | Séquence 1 | Configuration 2
Lancement de l'extraction sur 300 images en parallèle...


Extraction Multicoeur: 100%|██████████| 300/300 [00:02<00:00, 100.25it/s]



🚀 Lancement : HD | Séquence 1 | Configuration 3
Lancement de l'extraction sur 300 images en parallèle...


...


🚀 Lancement : HD | Séquence 3 | Configuration 2
Lancement de l'extraction sur 300 images en parallèle...


Extraction Multicoeur: 100%|██████████| 300/300 [00:03<00:00, 90.47it/s] 



🚀 Lancement : HD | Séquence 3 | Configuration 3
Lancement de l'extraction sur 300 images en parallèle...


Extraction Multicoeur: 100%|██████████| 300/300 [00:02<00:00, 112.74it/s]



✅ Extraction terminée !
Forme brute Train (Seq 1 & 2) : (2304000, 5)
Forme brute Test (Seq 3)       : (1152000, 5)
Train équilibré : 27264 défauts et 54528 saines.
Shape finale X_train : (81792, 5)
Shape finale X_test  : (1152000, 5)

📊 RÉSULTATS DU MODÈLE XGBOOST (Évalué uniquement sur la Séquence 3) :
              precision    recall  f1-score   support

           0       0.99      0.75      0.85   1137384
           1       0.03      0.55      0.05     14616

    accuracy                           0.75   1152000
   macro avg       0.51      0.65      0.45   1152000
weighted avg       0.98      0.75      0.84   1152000

💾 Succès : Modèle sauvegardé sous new_models/xgboost_hd_baseline.json !


In [8]:
import numpy as np
import pandas as pd
import xgboost as xgb
import traceback
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm # Import modifié pour éviter l'erreur IProgress !

# =========================================================================
# 1. PARAMÈTRES DU SUPER DATASET
# =========================================================================
CAMERA_TYPE = 'VGA'  # Change ceci en 'HD', 'VGA' ou 'SXGA'

# Création automatique du nouveau dossier demandé
os.makedirs('new_models', exist_ok=True)

dyn_mapping = {
    1: 'low dyn with columns 1',
    2: 'low dyn with columns 2',
    3: 'low dyn with columns 3'
}

# On prépare deux listes distinctes pour le Train et le Test
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

noms_colonnes = [
    'ecart_lt_moy', 'ratio_lt_moy', 
    'spatial_diff', 'diff_temp_absolue', 'scintillement_temporel', 
    # 'mean', 'energy'
]

# =========================================================================
# 2. BOUCLE D'EXTRACTION AUTOMATIQUE (Séparation Séquences 1&2 vs 3)
# =========================================================================
for seq in [1, 2, 3]:
    for config in [1, 2, 3]:
        print(f"\n🚀 Lancement : {CAMERA_TYPE} | Séquence {seq} | Configuration {config}")
        
        sequence_name = f'sequence_{seq}'
        dyn_name = dyn_mapping[config]
        
        dossier_json = r"C:\Users\alexc\Documents\Data_challenge\Data_Challenge\results"
        json_path = os.path.join(dossier_json, f"{CAMERA_TYPE}_{sequence_name}_config_{config}.json") 
        
        try:
            images = load_images(type=CAMERA_TYPE, sequence=sequence_name, dyn=dyn_name)
            json_data = load_json(json_path)
            
            X_batch, y_batch = build_dataset(images, json_data)
            
            # --- LE SCRIPT SE PARE ICI ---
            if seq in [1, 2]:
                X_train_list.append(X_batch)
                y_train_list.append(y_batch)
            else:  # seq == 3
                X_test_list.append(X_batch)
                y_test_list.append(y_batch)
            
        except Exception as e:
            print(f"⚠️ Erreur pour Seq {seq} / Config {config} : {e}")
            continue

# Fusion des blocs
X_train_raw = np.vstack(X_train_list)
y_train_raw = np.concatenate(y_train_list)

X_test_raw = np.vstack(X_test_list)
y_test_raw = np.concatenate(y_test_list)

print(f"\n✅ Extraction terminée !")
print(f"Forme brute Train (Seq 1 & 2) : {X_train_raw.shape}")
print(f"Forme brute Test (Seq 3)       : {X_test_raw.shape}")
# ==# =========================================================================
# 3. RÉÉQUILIBRAGE DU DATASET D'ENTRAÎNEMENT UNIQUEMENT
# =========================================================================
indices_defauts = np.where(y_train_raw == 1)[0]
indices_sains = np.where(y_train_raw == 0)[0]

# On garde 2 fois plus de colonnes saines que de défauts
nb_sains_a_garder = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_sains_a_garder, replace=False)

indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_train_balanced = X_train_raw[indices_finaux]
y_train_balanced = y_train_raw[indices_finaux]

print(f"Train équilibré : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} saines.")

# =========================================================================
# 4. ENTRAÎNEMENT ET SAUVEGARDE DANS 'new_models'
# =========================================================================
# Conversion en DataFrame Pandas
X_train = pd.DataFrame(X_train_balanced, columns=noms_colonnes)
X_test = pd.DataFrame(X_test_raw, columns=noms_colonnes)
y_train = y_train_balanced
y_test = y_test_raw

print(f"Shape finale X_train : {X_train.shape}")
print(f"Shape finale X_test  : {X_test.shape}")

# Calcul du ratio de poids pour XGBoost
ratio_poids = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

clf = xgb.XGBClassifier(
    n_estimators=150,
    scale_pos_weight=ratio_poids, 
    random_state=42,
    tree_method="hist",
    device="cpu"  # Mets "cuda" si ton GPU est disponible
)

# Entraînement sur Séquences 1 & 2
clf.fit(X_train, y_train)

# Prédiction sur la Séquence 3 (Inconnue du modèle)
y_pred = clf.predict(X_test)

print("\n📊 RÉSULTATS DU MODÈLE XGBOOST (Évalué uniquement sur la Séquence 3) :")
print(classification_report(y_test, y_pred))

# Sauvegarde dans le NOUVEAU dossier demandé
model_path = f'new_models/xgboost_{CAMERA_TYPE.lower()}_baseline.json'
clf.save_model(model_path)
print(f"💾 Succès : Modèle sauvegardé sous {model_path} !")


🚀 Lancement : VGA | Séquence 1 | Configuration 1
Lancement de l'extraction sur 562 images en parallèle...


Extraction Multicoeur: 100%|██████████| 562/562 [00:01<00:00, 299.46it/s]



🚀 Lancement : VGA | Séquence 1 | Configuration 2
Lancement de l'extraction sur 562 images en parallèle...


Extraction Multicoeur: 100%|██████████| 562/562 [00:01<00:00, 298.67it/s]



🚀 Lancement : VGA | Séquence 1 | Configuration 3
Lancement de l'extraction sur 562 images en parallèle...


...


🚀 Lancement : VGA | Séquence 3 | Configuration 2
Lancement de l'extraction sur 500 images en parallèle...


Extraction Multicoeur: 100%|██████████| 500/500 [00:02<00:00, 244.69it/s]



🚀 Lancement : VGA | Séquence 3 | Configuration 3
Lancement de l'extraction sur 500 images en parallèle...


Extraction Multicoeur: 100%|██████████| 500/500 [00:01<00:00, 257.49it/s]



✅ Extraction terminée !
Forme brute Train (Seq 1 & 2) : (2858880, 5)
Forme brute Test (Seq 3)       : (960000, 5)
Train équilibré : 19287 défauts et 38574 saines.
Shape finale X_train : (57861, 5)
Shape finale X_test  : (960000, 5)

📊 RÉSULTATS DU MODÈLE XGBOOST (Évalué uniquement sur la Séquence 3) :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    954665
           1       0.63      0.87      0.73      5335

    accuracy                           1.00    960000
   macro avg       0.81      0.94      0.86    960000
weighted avg       1.00      1.00      1.00    960000

💾 Succès : Modèle sauvegardé sous new_models/xgboost_vga_baseline.json !


In [9]:
import numpy as np
import pandas as pd
import xgboost as xgb
import traceback
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm # Import modifié pour éviter l'erreur IProgress !

# =========================================================================
# 1. PARAMÈTRES DU SUPER DATASET
# =========================================================================
CAMERA_TYPE = 'SXGA'  # Change ceci en 'HD', 'VGA' ou 'SXGA'

# Création automatique du nouveau dossier demandé
os.makedirs('new_models', exist_ok=True)

dyn_mapping = {
    1: 'low dyn with columns 1',
    2: 'low dyn with columns 2',
    3: 'low dyn with columns 3'
}

# On prépare deux listes distinctes pour le Train et le Test
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

noms_colonnes = [
    'ecart_lt_moy', 'ratio_lt_moy', 
    'spatial_diff', 'diff_temp_absolue', 'scintillement_temporel', 
    # 'mean', 'energy'
]

# =========================================================================
# 2. BOUCLE D'EXTRACTION AUTOMATIQUE (Séparation Séquences 1&2 vs 3)
# =========================================================================
for seq in [1, 2, 3]:
    for config in [1, 2, 3]:
        print(f"\n🚀 Lancement : {CAMERA_TYPE} | Séquence {seq} | Configuration {config}")
        
        sequence_name = f'sequence_{seq}'
        dyn_name = dyn_mapping[config]
        
        dossier_json = r"C:\Users\alexc\Documents\Data_challenge\Data_Challenge\results"
        json_path = os.path.join(dossier_json, f"{CAMERA_TYPE}_{sequence_name}_config_{config}.json") 
        
        try:
            images = load_images(type=CAMERA_TYPE, sequence=sequence_name, dyn=dyn_name)
            json_data = load_json(json_path)
            
            X_batch, y_batch = build_dataset(images, json_data)
            
            # --- LE SCRIPT SE PARE ICI ---
            if seq in [1, 2]:
                X_train_list.append(X_batch)
                y_train_list.append(y_batch)
            else:  # seq == 3
                X_test_list.append(X_batch)
                y_test_list.append(y_batch)
            
        except Exception as e:
            print(f"⚠️ Erreur pour Seq {seq} / Config {config} : {e}")
            continue

# Fusion des blocs
X_train_raw = np.vstack(X_train_list)
y_train_raw = np.concatenate(y_train_list)

X_test_raw = np.vstack(X_test_list)
y_test_raw = np.concatenate(y_test_list)

print(f"\n✅ Extraction terminée !")
print(f"Forme brute Train (Seq 1 & 2) : {X_train_raw.shape}")
print(f"Forme brute Test (Seq 3)       : {X_test_raw.shape}")
# ==# =========================================================================
# 3. RÉÉQUILIBRAGE DU DATASET D'ENTRAÎNEMENT UNIQUEMENT
# =========================================================================
indices_defauts = np.where(y_train_raw == 1)[0]
indices_sains = np.where(y_train_raw == 0)[0]

# On garde 2 fois plus de colonnes saines que de défauts
nb_sains_a_garder = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_sains_a_garder, replace=False)

indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_train_balanced = X_train_raw[indices_finaux]
y_train_balanced = y_train_raw[indices_finaux]

print(f"Train équilibré : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} saines.")

# =========================================================================
# 4. ENTRAÎNEMENT ET SAUVEGARDE DANS 'new_models'
# =========================================================================
# Conversion en DataFrame Pandas
X_train = pd.DataFrame(X_train_balanced, columns=noms_colonnes)
X_test = pd.DataFrame(X_test_raw, columns=noms_colonnes)
y_train = y_train_balanced
y_test = y_test_raw

print(f"Shape finale X_train : {X_train.shape}")
print(f"Shape finale X_test  : {X_test.shape}")

# Calcul du ratio de poids pour XGBoost
ratio_poids = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

clf = xgb.XGBClassifier(
    n_estimators=150,
    scale_pos_weight=ratio_poids, 
    random_state=42,
    tree_method="hist",
    device="cpu"  # Mets "cuda" si ton GPU est disponible
)

# Entraînement sur Séquences 1 & 2
clf.fit(X_train, y_train)

# Prédiction sur la Séquence 3 (Inconnue du modèle)
y_pred = clf.predict(X_test)

print("\n📊 RÉSULTATS DU MODÈLE XGBOOST (Évalué uniquement sur la Séquence 3) :")
print(classification_report(y_test, y_pred))

# Sauvegarde dans le NOUVEAU dossier demandé
model_path = f'new_models/xgboost_{CAMERA_TYPE.lower()}_baseline.json'
clf.save_model(model_path)
print(f"💾 Succès : Modèle sauvegardé sous {model_path} !")


🚀 Lancement : SXGA | Séquence 1 | Configuration 1
Lancement de l'extraction sur 500 images en parallèle...


Extraction Multicoeur: 100%|██████████| 500/500 [00:08<00:00, 60.38it/s]



🚀 Lancement : SXGA | Séquence 1 | Configuration 2
Lancement de l'extraction sur 500 images en parallèle...


Extraction Multicoeur: 100%|██████████| 500/500 [00:07<00:00, 70.28it/s]



🚀 Lancement : SXGA | Séquence 1 | Configuration 3
Lancement de l'extraction sur 500 images en parallèle...


...


🚀 Lancement : SXGA | Séquence 3 | Configuration 2
Lancement de l'extraction sur 500 images en parallèle...


Extraction Multicoeur: 100%|██████████| 500/500 [00:07<00:00, 70.41it/s]



🚀 Lancement : SXGA | Séquence 3 | Configuration 3
Lancement de l'extraction sur 500 images en parallèle...


Extraction Multicoeur: 100%|██████████| 500/500 [00:06<00:00, 76.42it/s]



✅ Extraction terminée !
Forme brute Train (Seq 1 & 2) : (4177920, 5)
Forme brute Test (Seq 3)       : (1920000, 5)
Train équilibré : 51848 défauts et 103696 saines.
Shape finale X_train : (155544, 5)
Shape finale X_test  : (1920000, 5)

📊 RÉSULTATS DU MODÈLE XGBOOST (Évalué uniquement sur la Séquence 3) :
              precision    recall  f1-score   support

           0       1.00      0.96      0.98   1897364
           1       0.19      0.74      0.31     22636

    accuracy                           0.96   1920000
   macro avg       0.60      0.85      0.64   1920000
weighted avg       0.99      0.96      0.97   1920000

💾 Succès : Modèle sauvegardé sous new_models/xgboost_sxga_baseline.json !
